In [ ]:
# ─── 1. Install ───────────────────────────────────────────────────────────────
!pip uninstall -y peft accelerate transformers -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  numpy==1.26.4 \
  Pillow

# ─── Restart runtime ──────────────────────────────────────────────────────────
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 16.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not

In [ ]:
# ─── 2. Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/ResNet18_IFND"
import os, time
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── 3. Imports ───────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import numpy as np
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from datasets import load_dataset, Image as HFImage
from transformers import get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 4. Load dataset ──────────────────────────────────────────────────────────
print("⏳ Loading dataset...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")
hf_dataset = hf_dataset.cast_column("image", HFImage(decode=False))
print(hf_dataset)

# ─── 5. Transform & Dataset Class ─────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class IFNDImageDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, transform):
        self.data      = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        label  = int(sample['label'])
        try:
            img_bytes = sample['image']['bytes']
            if img_bytes:
                image = Image.open(BytesIO(img_bytes)).convert("RGB")
            else:
                image = Image.new("RGB", (224, 224), (128, 128, 128))
        except Exception:
            image = Image.new("RGB", (224, 224), (128, 128, 128))
        return self.transform(image), torch.tensor(label, dtype=torch.long)

train_dataset = IFNDImageDataset(hf_dataset['train'],      transform)
val_dataset   = IFNDImageDataset(hf_dataset['validation'], transform)
test_dataset  = IFNDImageDataset(hf_dataset['test'],       transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 6. Model ─────────────────────────────────────────────────────────────────
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params/1e6:.2f}M")
print(f"Trainable params: {trainable_params/1e6:.2f}M")

# ─── 7. Training setup ────────────────────────────────────────────────────────
criterion    = nn.CrossEntropyLoss()
optimizer    = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps  = len(train_loader) * 5
warmup_steps = int(total_steps * 0.1)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps,
    num_training_steps=total_steps)

# ─── 8. Train & Eval functions ────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(device)
            outputs = model(images)
            probs   = torch.softmax(outputs, dim=1).cpu().numpy()
            preds   = np.argmax(probs, axis=1)
            all_probs.extend(probs[:, 1])
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary")
    return {
        "accuracy":  accuracy_score(all_labels, all_preds),
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       roc_auc_score(all_labels, all_probs)
    }

# ─── 9. Training loop ─────────────────────────────────────────────────────────
best_f1   = 0
patience  = 0
best_path = os.path.join(SAVE_DIR, "best_model.pth")

print("\n🆕 Training from scratch")
train_start = time.time()  # ← bắt đầu đo

for epoch in range(1, 6):
    epoch_start = time.time()
    train_loss  = train_epoch(model, train_loader, optimizer, scheduler, criterion)
    val_metrics = evaluate(model, val_loader)
    epoch_time  = (time.time() - epoch_start) / 60

    print(f"Epoch {epoch}/5 | Loss: {train_loss:.4f} | "
          f"F1: {val_metrics['f1']*100:.2f}% | "
          f"Acc: {val_metrics['accuracy']*100:.2f}% | "
          f"AUC: {val_metrics['auc']:.5f} | "
          f"Time: {epoch_time:.1f}min")

    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        torch.save(model.state_dict(), best_path)
        print(f"  ✅ Best model saved (F1={best_f1*100:.2f}%)")
        patience = 0
    else:
        patience += 1
        if patience >= 2:
            print(f"  ⏹ Early stopping at epoch {epoch}")
            break

total_train_time = (time.time() - train_start) / 3600  # giờ
print(f"\n⏱ Total training time: {total_train_time:.2f}h")

# ─── 10. Load best & Test ─────────────────────────────────────────────────────
model.load_state_dict(torch.load(best_path))
print("\n📊 Evaluating on test set...")
test_metrics = evaluate(model, test_loader)

print(f"\n{'='*40}")
print(f"Test Accuracy : {test_metrics['accuracy']*100:.2f}%")
print(f"Test Precision: {test_metrics['precision']*100:.2f}%")
print(f"Test Recall   : {test_metrics['recall']*100:.2f}%")
print(f"Test F1       : {test_metrics['f1']*100:.2f}%")
print(f"Test AUC      : {test_metrics['auc']:.5f}")
print(f"{'='*40}")

# ─── 11. Latency + VRAM ───────────────────────────────────────────────────────
model.eval()
dummy = torch.randn(1, 3, 224, 224).to(device)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

with torch.no_grad():
    for _ in range(10):
        model(dummy)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(dummy)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
vram = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0

print(f"\n{'='*40}")
print(f"Total params      : {total_params:,}")
print(f"Trainable params  : {trainable_params:,}")
print(f"Training time     : {total_train_time:.2f}h")
print(f"Latency median    : {np.median(latencies):.2f} ms")
print(f"VRAM (peak)       : {vram:.1f} MB")
print(f"{'='*40}")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda
⏳ Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1052 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1053 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 8416
    })
    validation: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1052
    })
    test: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1053
    })
})
Train: 8416 | Val: 1052 | Test: 1053
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 121MB/s]


Total params    : 11.18M
Trainable params: 11.18M

🆕 Training from scratch
Epoch 1/5 | Loss: 0.5270 | F1: 88.53% | Acc: 80.51% | AUC: 0.69556 | Time: 1.7min
  ✅ Best model saved (F1=88.53%)
Epoch 2/5 | Loss: 0.4035 | F1: 87.57% | Acc: 79.47% | AUC: 0.73702 | Time: 1.7min
Epoch 3/5 | Loss: 0.3167 | F1: 87.75% | Acc: 79.75% | AUC: 0.75222 | Time: 1.7min
  ⏹ Early stopping at epoch 3

⏱ Total training time: 0.09h

📊 Evaluating on test set...

Test Accuracy : 79.77%
Test Precision: 80.49%
Test Recall   : 97.15%
Test F1       : 88.04%
Test AUC      : 0.70419

Total params      : 11,177,538
Trainable params  : 11,177,538
Training time     : 0.09h
Latency median    : 2.80 ms
VRAM (peak)       : 217.4 MB
